In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import mplfinance as mpf

In [21]:
stock_selected="ZOMATO.NS"
#Reliance chosen in testing mode.The stock or stocks will change according to the Trading strategy. 

In [22]:
stock_data=yf.download(stock_selected,start="2023-08-15",end="2023-10-11", interval="5m")

[*********************100%%**********************]  1 of 1 completed


In [23]:
stock_data

,Open,High,Low,Close,Adj Close,Volume
Datetime,,,,,,
2023-08-16 09:15:00+05:30,92.300003,92.650002,91.699997,92.500000,92.500000,0
2023-08-16 09:20:00+05:30,92.500000,93.900002,92.449997,93.699997,93.699997,5389317
2023-08-16 09:25:00+05:30,93.849998,94.500000,93.650002,93.750000,93.750000,3462053
2023-08-16 09:30:00+05:30,93.800003,93.949997,93.699997,93.699997,93.699997,1366886
2023-08-16 09:35:00+05:30,93.750000,94.449997,93.599998,94.349998,94.349998,1879820
...,...,...,...,...,...,...
2023-10-10 15:05:00+05:30,106.099998,106.199997,106.050003,106.150002,106.150002,432123
2023-10-10 15:10:00+05:30,106.099998,106.400002,106.099998,106.150002,106.150002,976247
2023-10-10 15:15:00+05:30,106.050003,106.199997,106.050003,106.199997,106.199997,1003051


In [35]:
import pandas_ta as ta

stock_data["VWAP"]=ta.vwap(stock_data.High, stock_data.Low, stock_data.Close, stock_data.Volume)
stock_data['RSI']=ta.rsi(stock_data.Close, length=20)
my_bbands = ta.bbands(stock_data.Close, length=20, std=2.0)
stock_data=stock_data.join(my_bbands)

C:\Users\Acer\AppData\Local\Temp\ipykernel_14936\3656827146.py:3: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



In [36]:
stock_data

,Open,High,Low,Close,Adj Close,Volume,VWAP,RSI,BBL_14_2.0,BBM_14_2.0,...,BBB_14_2.0,BBP_14_2.0,VWAPSignal,TotalSignal,pointposbreak,BBL_20_2.0,BBM_20_2.0,BBU_20_2.0,BBB_20_2.0,BBP_20_2.0
Datetime,,,,,,,,,,,,,,,,,,,,,
2023-08-16 09:15:00+05:30,92.300003,92.650002,91.699997,92.500000,92.500000,0,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:20:00+05:30,92.500000,93.900002,92.449997,93.699997,93.699997,5389317,93.349998,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:25:00+05:30,93.849998,94.500000,93.650002,93.750000,93.750000,3462053,93.591197,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:30:00+05:30,93.800003,93.949997,93.699997,93.699997,93.699997,1366886,93.616899,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:35:00+05:30,93.750000,94.449997,93.599998,94.349998,94.349998,1879820,93.697143,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-10-10 15:05:00+05:30,106.099998,106.199997,106.050003,106.150002,106.150002,432123,106.313729,53.423186,105.724739,106.092857,...,0.693953,0.577617,0,0,NaN,105.648865,106.0900,106.531134,0.831623,0.568008
2023-10-10 15:10:00+05:30,106.099998,106.400002,106.099998,106.150002,106.150002,976247,106.312014,53.423186,105.787040,106.117857,...,0.623489,0.548584,0,0,NaN,105.654089,106.0800,106.505911,0.802999,0.582179
2023-10-10 15:15:00+05:30,106.050003,106.199997,106.050003,106.199997,106.199997,1003051,106.309126,54.319437,105.845589,106.142857,...,0.560127,0.596109,0,0,NaN,105.654089,106.0800,106.505911,0.802999,0.640871


In [37]:
VWAPsignal = [0]*len(stock_data)
backcandles = 15

for row in range(backcandles, len(stock_data)):
    upt = 1
    dnt = 1
    for i in range(row-backcandles, row+1):
        if max(stock_data.Open[i], stock_data.Close[i])>=stock_data.VWAP[i]:
            dnt=0
        if min( stock_data.Open[i], stock_data.Close[i])<= stock_data.VWAP[i]:
            upt=0
    if upt==1 and dnt==1:
        VWAPsignal[row]=3
    elif upt==1:
        VWAPsignal[row]=2
    elif dnt==1:
        VWAPsignal[row]=1

stock_data['VWAPSignal'] = VWAPsignal

C:\Users\Acer\AppData\Local\Temp\ipykernel_14936\639419903.py:8: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\Acer\AppData\Local\Temp\ipykernel_14936\639419903.py:10: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



In [38]:
stock_data

,Open,High,Low,Close,Adj Close,Volume,VWAP,RSI,BBL_14_2.0,BBM_14_2.0,...,BBB_14_2.0,BBP_14_2.0,VWAPSignal,TotalSignal,pointposbreak,BBL_20_2.0,BBM_20_2.0,BBU_20_2.0,BBB_20_2.0,BBP_20_2.0
Datetime,,,,,,,,,,,,,,,,,,,,,
2023-08-16 09:15:00+05:30,92.300003,92.650002,91.699997,92.500000,92.500000,0,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:20:00+05:30,92.500000,93.900002,92.449997,93.699997,93.699997,5389317,93.349998,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:25:00+05:30,93.849998,94.500000,93.650002,93.750000,93.750000,3462053,93.591197,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:30:00+05:30,93.800003,93.949997,93.699997,93.699997,93.699997,1366886,93.616899,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:35:00+05:30,93.750000,94.449997,93.599998,94.349998,94.349998,1879820,93.697143,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-10-10 15:05:00+05:30,106.099998,106.199997,106.050003,106.150002,106.150002,432123,106.313729,53.423186,105.724739,106.092857,...,0.693953,0.577617,0,0,NaN,105.648865,106.0900,106.531134,0.831623,0.568008
2023-10-10 15:10:00+05:30,106.099998,106.400002,106.099998,106.150002,106.150002,976247,106.312014,53.423186,105.787040,106.117857,...,0.623489,0.548584,0,0,NaN,105.654089,106.0800,106.505911,0.802999,0.582179
2023-10-10 15:15:00+05:30,106.050003,106.199997,106.050003,106.199997,106.199997,1003051,106.309126,54.319437,105.845589,106.142857,...,0.560127,0.596109,0,0,NaN,105.654089,106.0800,106.505911,0.802999,0.640871


In [39]:
def TotalSignal(l):
    if (stock_data.VWAPSignal[l]==2
        and stock_data.Close[l]<=stock_data['BBL_14_2.0'][l]
        and stock_data.RSI[l]<45):
            return 2
    if (stock_data.VWAPSignal[l]==1
        and stock_data.Close[l]>=stock_data['BBU_14_2.0'][l]
        and stock_data.RSI[l]>55):
            return 1
    return 0
        
TotSignal = [0]*len(stock_data)
for row in range(backcandles, len(stock_data)): 
    TotSignal[row] = TotalSignal(row)
stock_data['TotalSignal'] = TotSignal

C:\Users\Acer\AppData\Local\Temp\ipykernel_14936\2866421752.py:2: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\Acer\AppData\Local\Temp\ipykernel_14936\2866421752.py:6: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\Acer\AppData\Local\Temp\ipykernel_14936\2866421752.py:7: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

C:\Users\Acer\AppData\Local\Temp\ipykernel_14936\2866421752.py:8: FutureWarning:

Series.__getitem__

In [40]:
stock_data

,Open,High,Low,Close,Adj Close,Volume,VWAP,RSI,BBL_14_2.0,BBM_14_2.0,...,BBB_14_2.0,BBP_14_2.0,VWAPSignal,TotalSignal,pointposbreak,BBL_20_2.0,BBM_20_2.0,BBU_20_2.0,BBB_20_2.0,BBP_20_2.0
Datetime,,,,,,,,,,,,,,,,,,,,,
2023-08-16 09:15:00+05:30,92.300003,92.650002,91.699997,92.500000,92.500000,0,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:20:00+05:30,92.500000,93.900002,92.449997,93.699997,93.699997,5389317,93.349998,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:25:00+05:30,93.849998,94.500000,93.650002,93.750000,93.750000,3462053,93.591197,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:30:00+05:30,93.800003,93.949997,93.699997,93.699997,93.699997,1366886,93.616899,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-16 09:35:00+05:30,93.750000,94.449997,93.599998,94.349998,94.349998,1879820,93.697143,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-10-10 15:05:00+05:30,106.099998,106.199997,106.050003,106.150002,106.150002,432123,106.313729,53.423186,105.724739,106.092857,...,0.693953,0.577617,0,0,NaN,105.648865,106.0900,106.531134,0.831623,0.568008
2023-10-10 15:10:00+05:30,106.099998,106.400002,106.099998,106.150002,106.150002,976247,106.312014,53.423186,105.787040,106.117857,...,0.623489,0.548584,0,0,NaN,105.654089,106.0800,106.505911,0.802999,0.582179
2023-10-10 15:15:00+05:30,106.050003,106.199997,106.050003,106.199997,106.199997,1003051,106.309126,54.319437,105.845589,106.142857,...,0.560127,0.596109,0,0,NaN,105.654089,106.0800,106.505911,0.802999,0.640871


In [41]:
stock_data[stock_data.TotalSignal!=0].count()

Open             13
High             13
Low              13
Close            13
Adj Close        13
Volume           13
VWAP              7
RSI              13
BBL_14_2.0       13
BBM_14_2.0       13
BBU_14_2.0       13
BBB_14_2.0       13
BBP_14_2.0       13
VWAPSignal       13
TotalSignal      13
pointposbreak    13
BBL_20_2.0       13
BBM_20_2.0       13
BBU_20_2.0       13
BBB_20_2.0       13
BBP_20_2.0       13
dtype: int64

In [42]:
def pointposbreak(x):
    if x['TotalSignal']==1:
        return x['High']+1e-4
    elif x['TotalSignal']==2:
        return x['Low']-1e-4
    else:
        return np.nan

stock_data['pointposbreak'] = stock_data.apply(lambda row: pointposbreak(row), axis=1)

In [43]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
st=10400
dfpl = stock_data[st:st+350]
dfpl.reset_index(inplace=True)
fig = go.Figure(data=[go.Candlestick(x=dfpl.index,
                open=dfpl['Open'],
                high=dfpl['High'],
                low=dfpl['Low'],
                close=dfpl['Close']),
                go.Scatter(x=dfpl.index, y=dfpl.VWAP, 
                           line=dict(color='blue', width=1), 
                           name="VWAP"), 
                go.Scatter(x=dfpl.index, y=dfpl['BBL_14_2.0'], 
                           line=dict(color='green', width=1), 
                           name="BBL"),
                go.Scatter(x=dfpl.index, y=dfpl['BBU_14_2.0'], 
                           line=dict(color='green', width=1), 
                           name="BBU")])

fig.add_scatter(x=dfpl.index, y=dfpl['pointposbreak'], mode="markers",
                marker=dict(size=10, color="MediumPurple"),
                name="Signal")
fig.show()

In [44]:
dfpl = stock_data.copy()
import pandas_ta as ta
dfpl['ATR']=ta.atr(dfpl.High, dfpl.Low, dfpl.Close, length=7)
#help(ta.atr)
def SIGNAL():
    return dfpl.TotalSignal

In [45]:
from backtesting import Strategy
from backtesting import Backtest

class MyStrat(Strategy):
    initsize = 0.99
    mysize = initsize
    def init(self):
        super().init()
        self.signal1 = self.I(SIGNAL)

    def next(self):
        super().next()
        slatr = 1.2*self.data.ATR[-1]
        TPSLRatio = 1.5

        if len(self.trades)>0:
            if self.trades[-1].is_long and self.data.RSI[-1]>=90:
                self.trades[-1].close()
            elif self.trades[-1].is_short and self.data.RSI[-1]<=10:
                self.trades[-1].close()
        
        if self.signal1==2 and len(self.trades)==0:
            sl1 = self.data.Close[-1] - slatr
            tp1 = self.data.Close[-1] + slatr*TPSLRatio
            self.buy(sl=sl1, tp=tp1, size=self.mysize)
        
        elif self.signal1==1 and len(self.trades)==0:         
            sl1 = self.data.Close[-1] + slatr
            tp1 = self.data.Close[-1] - slatr*TPSLRatio
            self.sell(sl=sl1, tp=tp1, size=self.mysize)

bt = Backtest(dfpl, MyStrat, cash=100, margin=1/10, commission=0.00)
stat = bt.run()
stat

C:\Users\Acer\AppData\Local\Temp\ipykernel_14936\2352820476.py:32: UserWarning:

Some prices are larger than initial cash value. Note that fractional trading is not supported. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (GH-134).



Start                     2023-08-16 09:15...
End                       2023-10-10 15:25...
Duration                     55 days 06:10:00
Exposure Time [%]                    3.789474
Equity Final [$]                   131.362384
Equity Peak [$]                    156.068874
Return [%]                          31.362384
Buy & Hold Return [%]               14.432431
Return (Ann.) [%]                  510.449773
Volatility (Ann.) [%]              414.846457
Sharpe Ratio                         1.230455
Sortino Ratio                       16.435172
Calmar Ratio                        25.432142
Max. Drawdown [%]                  -20.071049
Avg. Drawdown [%]                   -5.744991
Max. Drawdown Duration       12 days 21:15:00
Avg. Drawdown Duration        2 days 11:54:00
# Trades                                   11
Win Rate [%]                        63.636364
Best Trade [%]                       1.275311
Worst Trade [%]                     -0.923077
Avg. Trade [%]                    